# Sanity test — pipeline on Gemma-2-2B-IT

Run this **before** paying for A100 hours on Gemma-2-9B-IT. Validates the FaithEval scoring stack end-to-end on a tiny subset, on a free Colab T4. Gemma-2-2B in bf16 needs ~5GB VRAM; fits easily.

Expected runtime: ~10 min on T4. Cost: $0.

If this works, the real M1/M2/M3 modules use `variant='primary'` and load Gemma-2-9B-IT on an A100.

## Cell 1 — env setup

On Colab: set HF_TOKEN and ANTHROPIC_API_KEY in Colab Secrets first.

In [ ]:
from google.colab import userdata
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
!git clone https://github.com/BraydenFeng/Algoverse.git || (cd Algoverse && git pull)
%cd Algoverse

!pip install -q -r requirements.txt


## Cell 2 — load Gemma-2-2B-IT (sanity variant)

In [ ]:
import sys
sys.path.insert(0, '.')

from src.lib.model_load import load_gemma

model, tokenizer = load_gemma(variant='sanity')
print(f'loaded {model.config._name_or_path}, n_layers={model.config.num_hidden_layers}, d_model={model.config.hidden_size}')

## Cell 3 — run 20 FaithEval prompts through the pipeline

In [ ]:
from pathlib import Path
from src.faitheval_eval import run_eval, summary

Path('outputs/sanity').mkdir(parents=True, exist_ok=True)
df = run_eval(
    model,
    tokenizer,
    limit=20,
    checkpoint_path=Path('outputs/sanity/faitheval_smoke.csv'),
    checkpoint_every=10,
)
df.head()

In [ ]:
summary(df)

## Cell 4 — generate 2 stories on Opus 4.7, sanity-check the generator

In [ ]:
from src.generate_stories import generate_one_story

for emotion in ['desperation', 'calm']:
    story = generate_one_story(emotion, context='a hospital waiting room late at night', target_words=200)
    print(f'\n=== {emotion.upper()} ===\n{story[:600]}')

In [ ]:
# 1. Inspect what Gemma actually said — labels alone hide the issue
  print("=== completions vs current labels ===")
  for _, row in df.head(20).iterrows():
      print(f"\n[{row['label']}/{row['method']}] qid={row['qid']}")
      print(f"  Q: {row['question'][:100]}")
      print(f"  A: {row['output'][:300]}")

  # 2. Re-classify the SAME outputs with force_judge=True (no model re-run, just re-label)
  # Cheaper than a full re-run: ~20 Haiku calls, pennies.
  from src.lib.classifier import classify
  from src.faitheval_eval import EvalRecord
  from dataclasses import asdict

  judge_labels = []
  for _, row in df.iterrows():
      r = classify(row['output'], row['question'], row['context'], force_judge=True)
      judge_labels.append({'qid': row['qid'], 'rule_label': row['label'], 'judge_label': r.label, 'judge_reason':
  r.reason})

  import pandas as pd
  diag = pd.DataFrame(judge_labels)
  print("\n=== rule vs judge ===")
  print(diag.head(20))
  print("\nRule label distribution:", df['label'].value_counts().to_dict())
  print("Judge label distribution:", diag['judge_label'].value_counts().to_dict())
  disagreements = (diag['rule_label'] != diag['judge_label']).sum()
  print(f"Disagreements: {disagreements}/{len(diag)}")

## What this validates

- HF login works and gated Gemma weights download (license accepted)
- Tokenization + greedy decode runs without OOM on T4
- FaithEval dataset loads from HF
- Rule-based classifier returns sensible labels
- Claude judge gets called on ambiguous cases and parses correctly
- Anthropic API access works for story generation
- Checkpoint CSV writes

## If this works

Open `m1_extract.ipynb`. Switch `variant='primary'` (Gemma-2-9B-IT), bring up an A100 box.

## If this fails

Fix here before paying for A100 hours. Common failures:
- Gemma license not accepted on HF → 403 on download
- ANTHROPIC_API_KEY missing → story generator + judge crash
- T4 OOM → unlikely on 2B model but reduce `max_new_tokens` if it happens